In [1]:
from torchgeo.trainers import PixelwiseRegressionTask
import torch
import pytorch_lightning as pl
import numpy as np
import rasterio
import cv2
import logging
from typing import List
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import torch.nn as nn
import os
from utils.data.LandsatDataModule import LandsatDataModule

'''
-Wandb
-By City, by geography default -> test wandb
-With/w/o pretrained weights
-Adding aug/reg
-Adding normalization
'''

os.environ["WANDB_NOTEBOOK_NAME"] = "TrainUNet-Basic.ipynb"

config = {
    "debug": False,
    "use_huggingface": False,
    "by_city": True,
    "learning_rate": 1e-4,
    "model": "unet",
    "backbone": "resnet50",
    "dataset": "pure_landsat",
    "epochs": 5000,
    "batch_size": 1,
    "pretrained_weights": True,
    "deterministic": True,
    "in_channels": 5
}

class LSTNowcaster(pl.LightningModule):
    def __init__(self, model="unet", backbone="resnet50", in_channels=5, learning_rate=1e-4, pretrained_weights=True):
        super().__init__()
        self.save_hyperparameters()
        self.model = PixelwiseRegressionTask(
            model=model,
            backbone=backbone,
            weights=pretrained_weights,
            in_channels=in_channels,
            num_outputs=1,
            loss="mse",
            lr=learning_rate
        )
        self.criterion = nn.MSELoss()
        self.learning_rate = learning_rate
        self.train_rmse = []
        self.test_rmse = []
        self.validate_rmse = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        loss = self.criterion(outputs[mask], targets[mask])
        self.save_rmse(batch, outputs, self.train_rmse)
        return {"loss": loss}
    
    def on_train_epoch_start(self):
        self.train_rmse = []

    def on_train_epoch_end(self):
        avg_rmse = torch.stack(self.train_rmse).mean()
        self.log("train_rmse_F", avg_rmse, 
             on_step=False,
             on_epoch=True,
             prog_bar=True,
             sync_dist=True)
    
    def save_rmse(self, batch, outputs, rmse_list):        
        targets = batch['target']
        mask = batch['mask']                
        
        mse_f = torch.mean((outputs[mask] - targets[mask])**2)
        rmse_f = torch.sqrt(mse_f)
                
        rmse_list.append(rmse_f)

    def validation_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        mse_loss = self.criterion(outputs[mask], targets[mask])
        self.save_rmse(batch, outputs, self.validate_rmse)
        return mse_loss
    
    def on_validation_epoch_start(self):
        self.validate_rmse = []

    def on_validation_epoch_end(self):
        avg_rmse = torch.stack(self.validate_rmse).mean()
        self.log("val_rmse_F", avg_rmse, prog_bar=True)

    def test_step(self, batch, batch_idx):
        inputs = batch['input']
        targets = batch['target']
        mask = batch['mask']

        outputs = self(inputs)
        mse_loss = self.criterion(outputs[mask], targets[mask])
        self.save_rmse(batch, outputs, self.test_rmse)
        return mse_loss
    
    def on_test_epoch_start(self):
        self.test_rmse = []

    def on_test_epoch_end(self):
        avg_rmse = torch.stack(self.test_rmse).mean()
        self.log("test_rmse_F", avg_rmse, prog_bar=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
    
wandb_logger = WandbLogger(
    project="heat-island",  # your project name
    name="unet-experiment",  # name of this particular run
    log_model=True,  # log model checkpoints
    save_code=True,
    save_dir="./wandb",  # where to save the logs locally
)
wandb_logger.log_hyperparams(config)
if config["dataset"] == "pure_landsat":
    data_module = LandsatDataModule(
        data_dir="./Data",
        batch_size=config["batch_size"],
        num_workers=5,
        byCity=config["by_city"],
        debug=config["debug"],
        useHuggingface = config["use_huggingface"]
    )
    data_module.setup()

# Initialize trainer with explicit steps
trainer = pl.Trainer(
    max_epochs=config["epochs"],
    gradient_clip_val=0.5,
    log_every_n_steps=10,
    enable_progress_bar=True,
    enable_model_summary=False,
    deterministic=config["deterministic"],
    num_sanity_val_steps=2,
    reload_dataloaders_every_n_epochs=1,
    logger=wandb_logger
)

wandb: Currently logged in as: jesus-guerrero (jesus-guerrero-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Preparing scene by city...: 100%|██████████| 7611/7611 [00:17<00:00, 429.77it/s]


Dataset splits - Train: 6079, Val: 682, Test: 850


/work/ubh496/.conda/envs/ml3/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /work/ubh496/.conda/envs/ml3/lib/python3.10/site-pac ...
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
model = LSTNowcaster(model=config["model"], backbone=config["backbone"], in_channels=config["in_channels"], learning_rate=config["learning_rate"], pretrained_weights=config["pretrained_weights"])

In [ ]:
trainer.fit(model=model, datamodule=data_module)

In [ ]:
trainer.test(model=model, datamodule=data_module)